In [1]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Read acceleration data
acceleration_data = pd.read_csv("../output/handshape_acceleration.csv")

acceleration_data.head()

,frame,time,acceleration
0,0,0.000000,0.011707
1,1,0.016667,0.011301
2,2,0.033333,0.010556
3,3,0.050000,0.009514
4,4,0.066667,0.008468


In [3]:
# Read segment data
segment_data = pd.read_csv("../output/handshape_segments.csv")

segment_data.head()

,start,end
0,0,90
1,98,220
2,226,276
3,281,289
4,293,316


In [4]:
# # Add hand_index data to acceleration_data (so add an index based on the hand and its)
# acceleration_data['hand_index'] = acceleration_data.groupby('hand').cumcount()

# acceleration_data.head()

In [9]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=acceleration_data["frame"],
        y=acceleration_data["acceleration"],
        mode="lines",
        line=dict(color="#1f77b4", width=1.5),
        name="Base",
        legendgroup="Base",
    )
)

colours = px.colors.qualitative.Plotly

for index, row in segment_data.iterrows():
    current_color = colours[index % len(colours)]

    segment_slice = acceleration_data[
        (acceleration_data["frame"] >= row["start"])
        & (acceleration_data["frame"] <= row["end"])
    ]

    if not segment_slice.empty:
        fig.add_trace(
            go.Scatter(
                x=segment_slice["frame"],
                y=segment_slice["acceleration"],
                mode="lines",
                line=dict(color=current_color, width=4),
                name=f"Seg {index}",
                legendgroup=f"Segment {index}",
                showlegend=True,
            )
        )

fig.update_layout(
    title_text="Hand Acceleration Data",
    height=600,
    showlegend=True,
    xaxis_title="Time (s)",
    yaxis_title="Acceleration (rad/s²)",
)

fig.show()

In [ ]:
# Plot acceleration data
fig = px.line(acceleration_data, x='time', y='acceleration', color='hand', title='Hand Acceleration Data')

frame_count = len(acceleration_data['time'].where(acceleration_data['hand'] == 'right').dropna())
t_min = acceleration_data['time'].min()
fps = frame_count / (acceleration_data['time'].max() - t_min)
fps = 30

colours = px.colors.qualitative.Plotly

# 3. Over de segmenten loopen en ze specifiek per hand inkleuren
for index, row in segment_data.iterrows():
    segment_hand = row['hand']
    current_color = colours[index % len(colours)]
    
    # Filter direct op de hand én de hand-specifieke indices (geen tijdsberekening nodig!)
    segment_slice = acceleration_data[
        (acceleration_data['hand'] == segment_hand) &
        (acceleration_data['hand_index'] >= row['start']) & 
        (acceleration_data['hand_index'] <= row['end'])
    ]
    
    # Plot het segment eroverheen
    if not segment_slice.empty:
        fig.add_trace(
            go.Scatter(
                x=segment_slice['time'],
                y=segment_slice['acceleration'],
                mode='lines',
                line=dict(color=current_color, width=4),
                name=f"Seg {index} ({segment_hand})",
                legendgroup=f"Segment {index}",
                showlegend=True
            )
        )

fig.show()